In [ ]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, roc_auc_score
)
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print('Libraries loaded successfully')

In [ ]:
# Primary dataset — Kaggle 100k rows
kaggle = pd.read_csv('../data/kaggle_diabetes.csv')
print('Kaggle dataset shape:', kaggle.shape)
print('Kaggle columns:', kaggle.columns.tolist())
print('Kaggle diabetes distribution:')
print(kaggle['diabetes'].value_counts())
print()

# Baseline dataset — Pima Indians
pima = pd.read_csv('../data/pima_diabetes.csv')
print('Pima dataset shape:', pima.shape)
print('Pima columns:', pima.columns.tolist())
print('Pima outcome distribution:')
print(pima['Outcome'].value_counts())

In [ ]:
df = kaggle.copy()

# Encode gender
df['gender_encoded'] = (df['gender'] == 'Male').astype(int)

# Encode smoking history
smoking_map = {
    'never': 0, 'No Info': 0, 'not current': 1,
    'ever': 1, 'former': 1, 'current': 2
}
df['smoking_encoded'] = df['smoking_history'].map(smoking_map).fillna(0)

# Convert binary diabetes outcome to 3-class risk
# LOW: no diabetes AND HbA1c < 5.7 AND glucose < 100
# HIGH: diabetes == 1 AND (HbA1c >= 6.5 OR glucose >= 126)
# MEDIUM: everything else

def assign_risk_kaggle(row):
    if row['diabetes'] == 0 and row['HbA1c_level'] < 5.7 and row['blood_glucose_level'] < 100:
        return 0  # LOW
    elif row['diabetes'] == 1 and (row['HbA1c_level'] >= 6.5 or row['blood_glucose_level'] >= 126):
        return 2  # HIGH
    else:
        return 1  # MEDIUM

df['risk_class'] = df.apply(assign_risk_kaggle, axis=1)

print('Risk class distribution:')
print(df['risk_class'].value_counts())
print(df['risk_class'].value_counts(normalize=True).round(3))

# Select features matching CLAUDE.md Section 7.4
# Note: we approximate missing features with available equivalents
KAGGLE_FEATURES = [
    'age',
    'gender_encoded',
    'blood_glucose_level',   # avg_glucose_30d proxy
    'blood_glucose_level',   # max_glucose_30d proxy (same column)
    'bmi',
    'HbA1c_level',
    'hypertension',          # systolic_bp proxy (1/0)
    'heart_disease',         # insulin_dependent proxy
    'smoking_encoded',       # avg_activity proxy (inverse)
]

# Rename to match our model feature names
df_model = pd.DataFrame({
    'age': df['age'],
    'gender': df['gender_encoded'],
    'avg_glucose_30d': df['blood_glucose_level'],
    'max_glucose_30d': df['blood_glucose_level'],
    'glucose_std': df['blood_glucose_level'] * 0.15,  # estimate 15% std
    'bmi': df['bmi'],
    'systolic_bp': 120 + (df['hypertension'] * 20),  # 120 or 140
    'hba1c': df['HbA1c_level'],
    'insulin_dependent': df['heart_disease'],
    'avg_activity': 1.5 - (df['smoking_encoded'] * 0.3),
    'readings_count': 15,  # assume average engagement
})

X_kaggle = df_model
y_kaggle = df['risk_class']

print('\nFinal feature matrix shape:', X_kaggle.shape)
print('Features:', X_kaggle.columns.tolist())

In [ ]:
pima_df = pima.copy()

# Replace zero values that are biologically impossible with median
cols_with_zeros = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
for col in cols_with_zeros:
    pima_df[col] = pima_df[col].replace(0, pima_df[col].median())

# Convert to 3-class risk
def assign_risk_pima(row):
    if row['Outcome'] == 0 and row['Glucose'] < 120:
        return 0  # LOW
    elif row['Outcome'] == 1 and row['Glucose'] >= 150:
        return 2  # HIGH
    else:
        return 1  # MEDIUM

pima_df['risk_class'] = pima_df.apply(assign_risk_pima, axis=1)

print('Pima risk class distribution:')
print(pima_df['risk_class'].value_counts())

# Map Pima columns to our feature names
pima_model = pd.DataFrame({
    'age': pima_df['Age'],
    'gender': 0,  # all female in Pima dataset
    'avg_glucose_30d': pima_df['Glucose'],
    'max_glucose_30d': pima_df['Glucose'],
    'glucose_std': pima_df['Glucose'] * 0.15,
    'bmi': pima_df['BMI'],
    'systolic_bp': pima_df['BloodPressure'],
    'hba1c': (pima_df['Glucose'] + 46.7) / 28.7,  # Nathan formula estimate
    'insulin_dependent': (pima_df['Insulin'] > 0).astype(int),
    'avg_activity': 1.0,  # unknown, use moderate default
    'readings_count': 10,
})

X_pima = pima_model
y_pima = pima_df['risk_class']

print('\nPima feature matrix shape:', X_pima.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_kaggle, y_kaggle,
    test_size=0.2,
    random_state=42,
    stratify=y_kaggle  # preserve class distribution in split
)

print(f'Training samples: {X_train.shape[0]}')
print(f'Test samples:     {X_test.shape[0]}')
print(f'Features:         {X_train.shape[1]}')
print()
print('Training class distribution:')
print(pd.Series(y_train).value_counts())

In [ ]:
print('Training Gradient Boosting Classifier...')

gb_model = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=5,
    min_samples_split=10,
    min_samples_leaf=4,
    subsample=0.8,
    random_state=42,
)

gb_model.fit(X_train, y_train)
print('Training complete')

In [ ]:
print('Training Logistic Regression baseline...')

lr_model = LogisticRegression(
    max_iter=1000,
    random_state=42,
    multi_class='multinomial',
)

lr_model.fit(X_train, y_train)
print('Training complete')

In [ ]:
labels = ['LOW (0)', 'MEDIUM (1)', 'HIGH (2)']

# Gradient Boosting results
gb_pred = gb_model.predict(X_test)
gb_acc = accuracy_score(y_test, gb_pred)

# Logistic Regression results
lr_pred = lr_model.predict(X_test)
lr_acc = accuracy_score(y_test, lr_pred)

print('=' * 50)
print('MODEL COMPARISON')
print('=' * 50)
print(f'Gradient Boosting Accuracy: {gb_acc:.4f} ({gb_acc*100:.1f}%)')
print(f'Logistic Regression Accuracy: {lr_acc:.4f} ({lr_acc*100:.1f}%)')
print()

print('--- Gradient Boosting Classification Report ---')
print(classification_report(y_test, gb_pred,
    target_names=['LOW', 'MEDIUM', 'HIGH']))

print('--- Logistic Regression Classification Report ---')
print(classification_report(y_test, lr_pred,
    target_names=['LOW', 'MEDIUM', 'HIGH']))

# Confusion matrix for primary model
print('Gradient Boosting Confusion Matrix:')
print('Rows = Actual, Columns = Predicted')
print('         LOW  MED  HIGH')
cm = confusion_matrix(y_test, gb_pred)
for i, row_label in enumerate(['LOW ', 'MED ', 'HIGH']):
    print(f'{row_label}  {cm[i]}')

# Clinical metric — recall on HIGH class is most important
gb_report = classification_report(y_test, gb_pred,
    target_names=['LOW', 'MEDIUM', 'HIGH'], output_dict=True)
high_recall = gb_report['HIGH']['recall']
print(f'\nHIGH risk recall: {high_recall:.4f} ({high_recall*100:.1f}%)')
print('(Target: > 85% — missing HIGH risk patients is dangerous)')

if gb_acc >= 0.80:
    print(f'\n✅ Accuracy target met: {gb_acc*100:.1f}% >= 80%')
else:
    print(f'\n⚠️  Accuracy below target: {gb_acc*100:.1f}% < 80%')

In [ ]:
print('Running 5-fold cross validation on Gradient Boosting...')

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(
    gb_model, X_kaggle, y_kaggle,
    cv=skf, scoring='accuracy', n_jobs=-1
)

print(f'CV Accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print(f'Individual folds: {cv_scores.round(4)}')

In [ ]:
importances = pd.Series(
    gb_model.feature_importances_,
    index=X_kaggle.columns
).sort_values(ascending=False)

print('Feature Importances (Gradient Boosting):')
print(importances.round(4))

In [ ]:
print('Validating on Pima Indians dataset (generalization test)...')

pima_pred = gb_model.predict(X_pima)
pima_acc = accuracy_score(y_pima, pima_pred)

print(f'Pima validation accuracy: {pima_acc:.4f} ({pima_acc*100:.1f}%)')
print(classification_report(y_pima, pima_pred,
    target_names=['LOW', 'MEDIUM', 'HIGH']))

In [ ]:
os.makedirs('../models', exist_ok=True)
model_path = '../models/risk_model.pkl'
joblib.dump(gb_model, model_path)

file_size = os.path.getsize(model_path) / (1024 * 1024)
print(f'Model saved to: {model_path}')
print(f'File size: {file_size:.1f} MB')

# Save feature config
risk_config = {
    'feature_cols': list(X_kaggle.columns),
    'target_col': 'risk_class',
    'classes': {0: 'LOW', 1: 'MEDIUM', 2: 'HIGH'},
    'model_version': '1.0.0',
    'accuracy': round(gb_acc, 4),
    'high_recall': round(high_recall, 4),
    'training_samples': X_train.shape[0],
}
joblib.dump(risk_config, '../models/risk_model_config.pkl')
print('Risk model config saved')

In [ ]:
loaded_risk = joblib.load(model_path)

# Simulate a high-risk Rwandan patient
test_patient = pd.DataFrame([[
    55,     # age
    1,      # gender (Male)
    185.0,  # avg_glucose_30d (high)
    230.0,  # max_glucose_30d (very high)
    28.5,   # glucose_std
    29.5,   # bmi (overweight)
    138,    # systolic_bp (elevated)
    8.2,    # hba1c (diabetic range)
    0,      # insulin_dependent
    0.8,    # avg_activity (low)
    12,     # readings_count
]], columns=X_kaggle.columns)

prediction = loaded_risk.predict(test_patient)[0]
probabilities = loaded_risk.predict_proba(test_patient)[0]
labels_map = {0: 'LOW', 1: 'MEDIUM', 2: 'HIGH'}

print('Test patient (high-risk Rwandan male, 55 years):')
print(f'  Avg glucose 30d: 185 mg/dL')
print(f'  HbA1c: 8.2%')
print(f'  BMI: 29.5')
print(f'  Blood pressure: 138 systolic')
print()
print(f'Predicted risk: {labels_map[prediction]}')
print(f'Probabilities:')
print(f'  LOW:    {probabilities[0]:.3f} ({probabilities[0]*100:.1f}%)')
print(f'  MEDIUM: {probabilities[1]:.3f} ({probabilities[1]*100:.1f}%)')
print(f'  HIGH:   {probabilities[2]:.3f} ({probabilities[2]*100:.1f}%)')
print(f'\nModel loaded and working correctly ✅')

print('\n' + '=' * 50)
print('STEP 3.4 COMPLETE')
print(f'Model file: ../models/risk_model.pkl')
print(f'Accuracy: {gb_acc*100:.1f}%')
print(f'HIGH risk recall: {high_recall*100:.1f}%')
print('Run Step 3.5 next to wire real models into Flask')
print('=' * 50)